# Movie Success Prediction - SQL Database Notebook

This notebook creates a relational SQLite database for the cleaned TMDB movie dataset. It separates the data into multiple tables, inserts the cleaned records, and runs SQL queries to support analysis for the final project.

In [1]:
import pandas as pd
import sqlite3
import os

df = pd.read_csv("../Datasets/clean_movies.csv")

df.head()

,id,title,genres,keywords,cast,budget,revenue,runtime,vote_average,vote_count,popularity,release_date,original_language,director,release_year,success,profit,roi,main_genre,main_actor
0,19995,Avatar,"['Action', 'Adventure', 'Fantasy', 'Science Fi...","['culture clash', 'future', 'space war', 'spac...","['Sam Worthington', 'Zoe Saldana', 'Sigourney ...",237000000,2787965087,162.0,7.2,11800,150.437577,2009-12-10,en,James Cameron,2009,True,2550965087,10.763566,Action,Sam Worthington
1,285,Pirates of the Caribbean: At World's End,"['Adventure', 'Fantasy', 'Action']","['ocean', 'drug abuse', 'exotic island', 'east...","['Johnny Depp', 'Orlando Bloom', 'Keira Knight...",300000000,961000000,169.0,6.9,4500,139.082615,2007-05-19,en,Gore Verbinski,2007,True,661000000,2.203333,Adventure,Johnny Depp
2,206647,Spectre,"['Action', 'Adventure', 'Crime']","['spy', 'based on novel', 'secret agent', 'seq...","['Daniel Craig', 'Christoph Waltz', 'Léa Seydo...",245000000,880674609,148.0,6.3,4466,107.376788,2015-10-26,en,Sam Mendes,2015,True,635674609,2.594590,Action,Daniel Craig
3,49026,The Dark Knight Rises,"['Action', 'Crime', 'Drama', 'Thriller']","['dc comics', 'crime fighter', 'terrorist', 's...","['Christian Bale', 'Michael Caine', 'Gary Oldm...",250000000,1084939099,165.0,7.6,9106,112.312950,2012-07-16,en,Christopher Nolan,2012,True,834939099,3.339756,Action,Christian Bale
4,49529,John Carter,"['Action', 'Adventure', 'Science Fiction']","['based on novel', 'mars', 'medallion', 'space...","['Taylor Kitsch', 'Lynn Collins', 'Samantha Mo...",260000000,284139100,132.0,6.1,2124,43.926995,2012-03-07,en,Andrew Stanton,2012,True,24139100,0.092843,Action,Taylor Kitsch


In [2]:
df.shape

(3227, 20)

In [3]:
os.makedirs("../Database", exist_ok=True)

conn = sqlite3.connect("../Database/movie_success.db")
cur = conn.cursor()

print("Database connected successfully.")

Database connected successfully.


In [4]:
cur.execute("""
CREATE TABLE IF NOT EXISTS movies (
    movie_id INTEGER PRIMARY KEY,
    title TEXT,
    release_year INTEGER,
    runtime REAL,
    original_language TEXT,
    main_genre TEXT
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS financials (
    movie_id INTEGER,
    budget INTEGER,
    revenue INTEGER,
    profit INTEGER,
    roi REAL,
    success BOOLEAN,
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS ratings (
    movie_id INTEGER,
    vote_average REAL,
    vote_count INTEGER,
    popularity REAL,
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS people (
    movie_id INTEGER,
    director TEXT,
    main_actor TEXT,
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
);
""")

conn.commit()

In [5]:
cur.execute("DELETE FROM people;")
cur.execute("DELETE FROM ratings;")
cur.execute("DELETE FROM financials;")
cur.execute("DELETE FROM movies;")
conn.commit()

In [6]:
movies_table = df[[
    "id", "title", "release_year", "runtime", "original_language", "main_genre"
]].copy()

movies_table.columns = [
    "movie_id", "title", "release_year", "runtime", "original_language", "main_genre"
]

movies_table.to_sql("movies", conn, if_exists="append", index=False)

3227

In [7]:
financials_table = df[[
    "id", "budget", "revenue", "profit", "roi", "success"
]].copy()

financials_table.columns = [
    "movie_id", "budget", "revenue", "profit", "roi", "success"
]

financials_table.to_sql("financials", conn, if_exists="append", index=False)

3227

In [8]:
ratings_table = df[[
    "id", "vote_average", "vote_count", "popularity"
]].copy()

ratings_table.columns = [
    "movie_id", "vote_average", "vote_count", "popularity"
]

ratings_table.to_sql("ratings", conn, if_exists="append", index=False)

3227

In [9]:
people_table = df[[
    "id", "director", "main_actor"
]].copy()

people_table.columns = [
    "movie_id", "director", "main_actor"
]

people_table.to_sql("people", conn, if_exists="append", index=False)

3227

In [10]:
for table in ["movies", "financials", "ratings", "people"]:
    count = pd.read_sql_query(f"SELECT COUNT(*) AS row_count FROM {table};", conn)
    print(table)
    display(count)

movies


,row_count
0,3227


financials


,row_count
0,3227


ratings


,row_count
0,3227


people


,row_count
0,3227


In [11]:
top_revenue = pd.read_sql_query("""
SELECT 
    m.title,
    m.release_year,
    f.budget,
    f.revenue,
    f.profit
FROM movies m
JOIN financials f
    ON m.movie_id = f.movie_id
ORDER BY f.revenue DESC
LIMIT 10;
""", conn)

top_revenue

,title,release_year,budget,revenue,profit
0,Avatar,2009,237000000,2787965087,2550965087
1,Titanic,1997,200000000,1845034188,1645034188
2,The Avengers,2012,220000000,1519557910,1299557910
3,Jurassic World,2015,150000000,1513528810,1363528810
4,Furious 7,2015,190000000,1506249360,1316249360
5,Avengers: Age of Ultron,2015,280000000,1405403694,1125403694
6,Frozen,2013,150000000,1274219009,1124219009
7,Iron Man 3,2013,200000000,1215439994,1015439994
8,Minions,2015,74000000,1156730962,1082730962
9,Captain America: Civil War,2016,250000000,1153304495,903304495


In [12]:
avg_profit_genre = pd.read_sql_query("""
SELECT 
    m.main_genre,
    COUNT(*) AS movie_count,
    ROUND(AVG(f.profit), 2) AS avg_profit,
    ROUND(AVG(f.roi), 2) AS avg_roi
FROM movies m
JOIN financials f
    ON m.movie_id = f.movie_id
GROUP BY m.main_genre
HAVING movie_count >= 20
ORDER BY avg_profit DESC;
""", conn)

avg_profit_genre

,main_genre,movie_count,avg_profit,avg_roi
0,Animation,98,2.185274e+08,6.28
1,Family,38,1.722063e+08,2.72
2,Adventure,288,1.707187e+08,4.71
3,Science Fiction,79,1.461393e+08,4.19
4,Fantasy,93,1.196426e+08,3.70
5,Action,588,9.732282e+07,1.88
6,Romance,70,6.988704e+07,3.43
7,Mystery,27,6.598520e+07,4.15
8,Thriller,118,6.087310e+07,2.94
9,Comedy,634,5.621024e+07,4.26


In [13]:
high_rated = pd.read_sql_query("""
SELECT 
    m.title,
    m.release_year,
    r.vote_average,
    r.vote_count,
    r.popularity
FROM movies m
JOIN ratings r
    ON m.movie_id = r.movie_id
WHERE r.vote_count >= 1000
ORDER BY r.vote_average DESC
LIMIT 10;
""", conn)

high_rated

,title,release_year,vote_average,vote_count,popularity
0,The Shawshank Redemption,1994,8.5,8205,136.747729
1,The Godfather,1972,8.4,5893,143.659698
2,Fight Club,1999,8.3,9413,146.757391
3,Schindler's List,1993,8.3,4329,104.469351
4,Spirited Away,2001,8.3,3840,118.968562
5,The Godfather: Part II,1974,8.3,3338,105.792936
6,Pulp Fiction,1994,8.3,8428,121.463076
7,Whiplash,2014,8.3,4254,192.528841
8,The Dark Knight,2008,8.2,12002,187.322927
9,The Green Mile,1999,8.2,4048,103.698022


In [14]:
director_success = pd.read_sql_query("""
SELECT 
    p.director,
    COUNT(*) AS movie_count,
    ROUND(AVG(f.profit), 2) AS avg_profit,
    ROUND(AVG(r.vote_average), 2) AS avg_rating
FROM people p
JOIN financials f
    ON p.movie_id = f.movie_id
JOIN ratings r
    ON p.movie_id = r.movie_id
GROUP BY p.director
HAVING movie_count >= 3
ORDER BY avg_profit DESC
LIMIT 10;
""", conn)

director_success

,director,movie_count,avg_profit,avg_rating
0,Joss Whedon,3,8.082770e+08,7.37
1,James Cameron,7,7.338099e+08,7.33
2,George Lucas,5,5.958674e+08,6.96
3,Peter Jackson,9,5.784048e+08,7.33
4,Pete Docter,3,5.635088e+08,7.73
5,David Yates,3,5.496383e+08,6.77
6,Carlos Saldanha,4,5.423629e+08,6.45
7,Anthony Russo,3,5.081675e+08,6.70
8,Andrew Adamson,4,4.894266e+08,6.75
9,Francis Lawrence,5,4.698914e+08,6.84


In [15]:
success_count = pd.read_sql_query("""
SELECT 
    success,
    COUNT(*) AS movie_count
FROM financials
GROUP BY success;
""", conn)

success_count

,success,movie_count
0,0,789
1,1,2438


In [16]:
# Create SQL_Results folder one level above Notebooks
os.makedirs("../SQL_Results", exist_ok=True)

top_revenue.to_csv("../SQL_Results/top_revenue_movies.csv", index=False)
avg_profit_genre.to_csv("../SQL_Results/avg_profit_by_genre.csv", index=False)
high_rated.to_csv("../SQL_Results/high_rated_movies.csv", index=False)
director_success.to_csv("../SQL_Results/director_success.csv", index=False)
success_count.to_csv("../SQL_Results/success_count.csv", index=False)

print("SQL result files saved successfully in Final Course Project/SQL_Results")

SQL result files saved successfully in Final Course Project/SQL_Results
